# 08 — Portfolio Construction

Use `src/portfolio/optimizer.py` and model forecasts to build optimal allocations.

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
PROC_DIR = ROOT / 'data' / 'processed'
REPORT_DIR = ROOT / 'outputs' / 'reports'
PORT_DIR = ROOT / 'outputs' / 'portfolio'
PORT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.preprocessor import SELECTED, NAMES

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

# ── Load ensemble forecasts (5-day ahead) ─────────────────────
print('Loading ensemble forecasts and historical data...')
forecast_files = [
    ROOT / 'outputs' / 'predictions' / 'arima_5day_forecasts.csv',
    ROOT / 'outputs' / 'predictions' / 'ets_5day_forecasts.csv',
    ROOT / 'outputs' / 'predictions' / 'prophet_5day_forecasts.csv',
    ROOT / 'outputs' / 'predictions' / 'lstm_5day_forecasts.csv',
]

ensemble_fc = None
for fpath in forecast_files:
    if fpath.exists():
        df = pd.read_csv(fpath)
        if 'Forecast' in df.columns and 'Ticker' in df.columns:
            if ensemble_fc is None:
                ensemble_fc = df[['Ticker', 'Forecast']].groupby('Ticker')['Forecast'].mean().reset_index()
                ensemble_fc.columns = ['Ticker', 'Mean_Forecast']
            else:
                temp = df[['Ticker', 'Forecast']].groupby('Ticker')['Forecast'].mean().reset_index()
                temp.columns = ['Ticker', 'Forecast_Model']
                ensemble_fc = ensemble_fc.merge(temp, on='Ticker', how='outer')

if ensemble_fc is None or ensemble_fc.empty:
    print('WARNING: Could not load forecasts. Using historical means for portfolio weighting.')
    ensemble_fc = pd.DataFrame({'Ticker': SELECTED})

# ── Load current prices and historical data ────────────────────
print('Loading historical test data...')
price_data = {}
for ticker in SELECTED:
    test_path = PROC_DIR / f'{ticker}_test_full.csv'
    if test_path.exists():
        df = pd.read_csv(test_path, index_col=0, parse_dates=True)
        if 'Close' in df.columns:
            price_data[ticker] = df['Close']

# Use last available price
current_prices = {t: price_data[t].iloc[-1] if t in price_data else np.nan 
                  for t in SELECTED}

# Load GARCH volatility
garch_vol = {}
garch_path = REPORT_DIR / 'garch_volatility_report.csv'
if garch_path.exists():
    garch_df = pd.read_csv(garch_path)
    garch_vol = dict(zip(garch_df['Ticker'], garch_df['Annualized_Volatility_%'] / 100))

# ── Expected return proxy (5-day forecast change) ──────────────
expected_returns = {}
for ticker in SELECTED:
    if ticker in current_prices and not np.isnan(current_prices[ticker]):
        curr_price = current_prices[ticker]
        if ticker in price_data and len(price_data[ticker]) > 0:
            # Use average of historical return and forecast-implied return
            hist_return = price_data[ticker].pct_change().mean()
            
            # If we have ensemble forecast, use it; otherwise fallback
            fc_rows = ensemble_fc[ensemble_fc['Ticker'] == ticker]
            if not fc_rows.empty and 'Mean_Forecast' in ensemble_fc.columns:
                fc_price = fc_rows['Mean_Forecast'].values[0]
            else:
                fc_price = curr_price * (1 + hist_return * 5)  # 5-day simple projection
            
            expected_returns[ticker] = (fc_price - curr_price) / curr_price
        else:
            expected_returns[ticker] = 0.0
    else:
        expected_returns[ticker] = 0.0

# ── Risk-adjusted expected return (reward / risk) ─────────────
risk_adjusted = {}
for ticker in SELECTED:
    vol = garch_vol.get(ticker, 0.25)  # Default 25% annualized vol
    ret = expected_returns.get(ticker, 0.0)
    # Avoid division by zero; use Sharpe-like ratio
    risk_adjusted[ticker] = ret / (vol + 1e-6) if vol > 0 else ret

# ── Portfolio Optimization (Mean-Variance with constraints) ────
print('Computing optimal portfolio weights...')

# Normalize risk-adjusted returns to probabilities
min_weight = 0.05
max_weight = 0.25
min_stocks = 5
max_stocks = len(SELECTED)

# Start with risk-adjusted allocation
ra_values = np.array([max(risk_adjusted.get(t, 0), 0) for t in SELECTED])
if ra_values.sum() > 0:
    weights = ra_values / ra_values.sum()
else:
    weights = np.ones(len(SELECTED)) / len(SELECTED)

# Apply min/max constraints
weights = np.clip(weights, min_weight, max_weight)
weights = weights / weights.sum()

# Count non-zero allocations
non_zero = (weights > min_weight).sum()
if non_zero < min_stocks:
    # Force allocation to min_stocks highest-scoring
    scores = np.argsort(ra_values)[-min_stocks:]
    weights = np.zeros(len(SELECTED))
    weights[scores] = 1.0 / min_stocks
elif non_zero > max_stocks:
    # Keep top max_stocks by score
    scores = np.argsort(ra_values)[-max_stocks:]
    weights = np.zeros(len(SELECTED))
    weights[scores] = 1.0 / max_stocks

# Ensure constraints
weights = np.clip(weights, min_weight, max_weight)
weights = weights / weights.sum()

# ── Build allocation table ────────────────────────────────────
total_capital = 1_000_000  # ₹10,00,000

allocation = []
for ticker, weight in zip(SELECTED, weights):
    curr_p = current_prices.get(ticker, np.nan)
    exp_r = expected_returns.get(ticker, 0.0)
    vol = garch_vol.get(ticker, 0.25)
    ra = risk_adjusted.get(ticker, 0.0)
    
    allocation_value = total_capital * weight
    shares = int(allocation_value / curr_p) if not np.isnan(curr_p) else 0
    actual_value = shares * curr_p if not np.isnan(curr_p) else 0
    actual_weight = actual_value / total_capital if total_capital > 0 else 0
    
    allocation.append({
        'Ticker': ticker,
        'Company': NAMES.get(ticker, ticker),
        'Weight_%': round(weight * 100, 2),
        'Allocation_₹': round(allocation_value, 0),
        'Current_Price_₹': round(curr_p, 2) if not np.isnan(curr_p) else np.nan,
        'Shares': shares,
        'Actual_Value_₹': round(actual_value, 0),
        'Actual_Weight_%': round(actual_weight * 100, 2),
        'Expected_Return_%_5D': round(exp_r * 100, 2),
        'Volatility_%': round(vol * 100, 2),
        'Risk_Adj_Score': round(ra, 4),
    })

port_df = pd.DataFrame(allocation)
port_df.to_csv(PORT_DIR / 'final_portfolio_allocation.csv', index=False)

print('Portfolio Allocation:')
display(port_df)

total_allocated = port_df['Actual_Value_₹'].sum()
print(f'\nTotal Capital Available: ₹{total_capital:,.0f}')
print(f'Total Allocated: ₹{total_allocated:,.0f}')
print(f'Remaining Cash: ₹{total_capital - total_allocated:,.0f}')

# ── Visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Allocation pie chart
ax = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, len(port_df)))
ax.pie(port_df['Actual_Weight_%'], labels=port_df['Ticker'], autopct='%1.1f%%',
       colors=colors, startangle=90)
ax.set_title('Portfolio Allocation by Weight')

# Risk vs Return scatter
ax = axes[1]
ax.scatter(port_df['Volatility_%'], port_df['Expected_Return_%_5D'] * 100,
           s=port_df['Actual_Weight_%'] * 10, alpha=0.6, c=colors)
for idx, row in port_df.iterrows():
    ax.annotate(row['Ticker'], 
                (row['Volatility_%'], row['Expected_Return_%_5D'] * 100),
                fontsize=9, ha='center')
ax.set_xlabel('Volatility (Annualized %)')
ax.set_ylabel('Expected Return (5-day %)')
ax.set_title('Risk-Return Profile')
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(PORT_DIR / 'portfolio_allocation_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Portfolio construction complete. Saved:')
print(f'   - {PORT_DIR / "final_portfolio_allocation.csv"}')
print(f'   - {PORT_DIR / "portfolio_allocation_chart.png"}')